# Manage Genie Space Permissions

This notebook manages permissions for a Databricks Genie space via the **REST API**.

**Workflow:**
1. **Review** current permissions on the Genie space (GET)
2. **Conditionally add** a user group with `CAN_MANAGE` rights if not already granted (PATCH)

> **Note:** Genie spaces use the `genie` object type in the Permissions API endpoint (`/api/2.0/permissions/genie/{id}`).

In [0]:
dbutils.widgets.text("genie_space_id", "", "Genie Space ID")
dbutils.widgets.text("group_name", "data-team", "Group Name (CAN_MANAGE)")

genie_space_id = dbutils.widgets.get("genie_space_id")
group_name = dbutils.widgets.get("group_name")

print(f"Genie Space ID : {genie_space_id}")
print(f"Group Name     : {group_name}")

## Step 1: Review Current Permissions

In [0]:
import requests, json

# --- Workspace URL and auth token ---
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
api_token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()

base_url = f"https://{workspace_url}/api/2.0/permissions/genie/{genie_space_id}"
headers = {"Authorization": f"Bearer {api_token}"}

# --- GET current permissions ---
response = requests.get(base_url, headers=headers)

if response.status_code != 200:
    print(f"ERROR {response.status_code}: {response.text}")
    raise SystemExit(f"Failed to fetch permissions (HTTP {response.status_code})")

permissions = response.json()
print("=== Raw Permissions JSON ===")
print(json.dumps(permissions, indent=2))

# --- Summary table ---
print("\n=== Permissions Summary ===")
print(f"{'Principal':<40} {'Permission Level':<20} {'Inherited'}")
print("-" * 80)

for acl in permissions.get("access_control_list", []):
    principal = (
        acl.get("user_name")
        or acl.get("group_name")
        or acl.get("service_principal_name")
        or "unknown"
    )
    for perm in acl.get("all_permissions", []):
        level = perm.get("permission_level", "N/A")
        inherited = perm.get("inherited", False)
        inherited_from = ""
        if inherited:
            sources = perm.get("inherited_from_object", [])
            if sources:
                inherited_from = f" (from {sources[0]})"
        print(f"{principal:<40} {level:<20} {inherited}{inherited_from}")

## Step 2: Add CAN_MANAGE for Group (if not already granted)

In [0]:
import requests, json

# --- Re-fetch current permissions ---
response = requests.get(base_url, headers=headers)

if response.status_code != 200:
    print(f"ERROR {response.status_code}: {response.text}")
    raise SystemExit(f"Failed to fetch permissions (HTTP {response.status_code})")

permissions = response.json()

# --- Check if the group already has CAN_MANAGE ---
already_granted = False
inherited_flag = ""

for acl in permissions.get("access_control_list", []):
    if acl.get("group_name") == group_name:
        for perm in acl.get("all_permissions", []):
            if perm.get("permission_level") == "CAN_MANAGE":
                already_granted = True
                inherited_flag = " (inherited)" if perm.get("inherited", False) else " (direct)"
                break
        break

if already_granted:
    print(f"Group '{group_name}' already has CAN_MANAGE{inherited_flag}. No update needed.")
else:
    # --- PATCH to add CAN_MANAGE ---
    patch_body = {
        "access_control_list": [
            {
                "group_name": group_name,
                "permission_level": "CAN_MANAGE"
            }
        ]
    }

    print(f"Granting CAN_MANAGE to group '{group_name}'...")
    patch_response = requests.patch(base_url, headers=headers, json=patch_body)

    if patch_response.status_code == 200:
        print("Success! Updated permissions:")
        print(json.dumps(patch_response.json(), indent=2))
    else:
        print(f"ERROR {patch_response.status_code}: {patch_response.text}")
        raise SystemExit(f"Failed to update permissions (HTTP {patch_response.status_code})")